# IOAI — 2025 National Byzantine Notation (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train_data.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-byzantine.zip', '/tmp/byz.zip')
    with zipfile.ZipFile('/tmp/byz.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train_data.csv' if os.path.exists('data/train_data.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cv2 as cv
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn.functional as F
import torch.nn as nn
import torch

In [ ]:
seed = 42

torch.random.manual_seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)

batch_size = 128

device = "cuda" if torch.cuda.is_available() else "cpu"

root_path = "data"   # 로컬 데이터(원본: /home/stefan/.../kits/dataset-...)

# Data preparation

In [ ]:
class BabilonianTrainDataset(Dataset):
    def __init__(self, df, indices=None, is_train: bool = True):
        self.df = df.iloc[indices].reset_index(drop=True) if indices is not None else df

        self.label2idx = {
            label: idx for idx, label in enumerate(self.df["Effect"].unique())
        }
        self.idx2label = {idx: label for label, idx in self.label2idx.items()}
        self.is_train = is_train

        self.transforms = transforms.Compose([
            transforms.Resize((48, 48))] + 
            ([transforms.RandAugment(num_ops=4, magnitude=11)] if self.is_train else []) + [
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5]),
        ])

    def get_labels(self, indices):
        return [self.idx2label[idx] for idx in indices]

    def __getitem__(self, idx):
        img_row = self.df.iloc[idx]
        img_path = f"{root_path}/{img_row['Path']}"
        img = Image.open(img_path).convert("L")

        img = self.transforms(img)

        label = self.label2idx[img_row["Effect"]]
        return img, torch.tensor(label, dtype=torch.long)

    def __len__(self):
        return len(self.df)

In [ ]:
df = pd.read_csv("data/train_data.csv")
df = df[df.index % 5 != 0].reset_index(drop=True)   # 누수 방지: eval_local 합성에 쓰인 held-out(index%5==0) 제외
train_indices, val_indices = train_test_split(
    range(len(df)), test_size=0.2, random_state=seed
)

train_dataset = BabilonianTrainDataset(df, indices=train_indices)
val_dataset = BabilonianTrainDataset(df, indices=val_indices, is_train=False)

# Share label mapping to ensure consistency
val_dataset.label2idx = train_dataset.label2idx
val_dataset.idx2label = train_dataset.idx2label

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# sanity check
img_batch, label_batch = next(iter(train_loader))
print(img_batch.shape)
img, label = img_batch[0].cpu(), label_batch[0]

print(label.item())
plt.imshow(img.permute(1, 2, 0))

# Model

In [ ]:
class BabilonianCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(8)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2, 2)
        
        self.fc1 = nn.LazyLinear(128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

In [ ]:
model = BabilonianCNN().to(device)

In [ ]:
# sanity check
test_input, _ = next(iter(train_loader))
print(model(test_input.to(device)).shape)

# Training

In [ ]:
lr = 1e-3
epochs = 150

In [ ]:
criterion = nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

In [ ]:
train_losses = []
val_losses = []
val_accs = []

for epoch in range(1, epochs+1):
    # Train
    model.train()
    running_loss = 0.0

    for img, label in train_loader:
        img, label = img.to(device), label.to(device)
        pred = model(img)
        loss = criterion(pred, label)

        optim.zero_grad()
        loss.backward()
        optim.step()

        running_loss += loss.item()

    train_losses.append(running_loss / len(train_loader))

    # Validate
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for img, label in val_loader:
            img, label = img.to(device), label.to(device)
            pred = model(img)
            loss = criterion(pred, label)
            val_loss += loss.item()

            correct += (pred.argmax(dim=1) == label).sum().item()
            total += label.size(0)

    val_losses.append(val_loss / len(val_loader))
    val_acc = correct / total
    val_accs.append(val_acc)

    if epoch % 5 == 0:
        print(
            f"Epoch {epoch}/{epochs} | Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} | Val Acc: {val_acc:.4f}"
        )

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3))

ax1.plot(range(epochs), train_losses, label="Train")
ax1.plot(range(epochs), val_losses, label="Validation")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True)

ax2.plot(range(epochs), val_accs)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.grid(True)

plt.tight_layout()
plt.show()

# Evaluation data preparation

In [ ]:
df_test = pd.read_csv("data/test_data.csv")

# read a grayscale (for debug) image, binarize it
sample_id = 151
path = f"{root_path}/{df_test.iloc[sample_id]['datapointID']}"
img = cv.imread(path, cv.IMREAD_GRAYSCALE)
plt.imshow(img)

In [ ]:
_, img = cv.threshold(img, 120, 255, cv.THRESH_BINARY)

# remove the background
blur = cv.GaussianBlur(img, (3, 3), cv.BORDER_DEFAULT)
thresh = cv.adaptiveThreshold(blur, 255, cv.ADAPTIVE_THRESH_MEAN_C, cv.THRESH_BINARY, 15, 10)
inverted = cv.bitwise_not(thresh)  # or just ~thresh

plt.imshow(inverted)

In [ ]:
# find the contours of the babilonian things
thresh = cv.adaptiveThreshold(img, 255, cv.ADAPTIVE_THRESH_MEAN_C, cv.THRESH_BINARY_INV, 15, 10)
contours, _ = cv.findContours(thresh, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

print(f"found {len(contours)} things")
for contour in contours:
   x, y, w, h = cv.boundingRect(contour)
   cv.rectangle(thresh, (x, y), (x + w, y + h), 255, 1)
plt.imshow(thresh)

In [ ]:
# just put togheter the stuff from above
def remove_background(img):
   _, img = cv.threshold(img, 120, 255, cv.THRESH_BINARY)
   blur = cv.GaussianBlur(img, (3, 3), cv.BORDER_DEFAULT)
   thresh = cv.adaptiveThreshold(blur, 255, cv.ADAPTIVE_THRESH_MEAN_C, cv.THRESH_BINARY, 15, 10)
   return thresh

def extract_patches(img, min_area=100, pad=2):
   img = remove_background(img)
   thresh = cv.adaptiveThreshold(img, 255, cv.ADAPTIVE_THRESH_MEAN_C, cv.THRESH_BINARY_INV, 15, 10)
   contours, _ = cv.findContours(thresh, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
   
   patches = []
   for cnt in contours:
       x, y, w, h = cv.boundingRect(cnt)
       if w * h < min_area:
           continue
       
       x_start = max(x - pad, 0)
       y_start = max(y - pad, 0)
       x_end = min(x + w + pad, img.shape[1])
       y_end = min(y + h + pad, img.shape[0])
       patch = img[y_start:y_end, x_start:x_end]
       patches.append((x_start, patch))
   
   # we want the patches to be in the order they appear in the original images
   patches = [patch for _, patch in sorted(patches, key=lambda p: p[0])]
   return patches

In [ ]:
path = f"{root_path}/{df_test.iloc[sample_id]['datapointID']}"
img = cv.imread(path, cv.IMREAD_GRAYSCALE)
patches = extract_patches(img)

cols = 6
rows = int(np.ceil((len(patches) + 1) / cols))
plt.figure(figsize=(15, 3 * rows))
plt.subplot(rows, cols, 1)
plt.title("Imagine originală")
plt.axis("off")
plt.imshow(img, cmap="gray")

for i, patch in enumerate(patches):
    plt.subplot(rows, cols, i + 2)
    plt.title(f"Patch {i + 1}")
    plt.axis("off")
    plt.imshow(patch, cmap="gray")
plt.tight_layout()
plt.show()

In [ ]:
class BabilonianTestDataset(Dataset):
    def __init__(self):
        self.df = pd.read_csv("data/test_data.csv")
        self.transforms = transforms.Compose(
            [
                transforms.ToPILImage(),
                transforms.Resize((48, 48)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5], std=[0.5]),
            ]
        )

    def __getitem__(self, idx):
        path = f"{root_path}/{self.df.iloc[idx]['datapointID']}"
        img = cv.imread(path, cv.IMREAD_GRAYSCALE)
        raw_patches = extract_patches(img)

        patches = [self.transforms(patch) for patch in raw_patches]
        patches = torch.stack(patches) if patches else torch.empty(0, 1, 48, 48)
        return patches, self.df.iloc[idx]["datapointID"]

    def __len__(self):
        return len(self.df)

In [ ]:
test_dataset = BabilonianTestDataset()
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)

In [ ]:
# sanity check
patches, did = next(iter(test_loader))
print(patches.shape, did)

# Submission

In [ ]:
def encode_prediction(predictions):
    """Encode sequence of predictions into submission format."""
    if not predictions:
        return ""

    result = []
    for i, pred in enumerate(predictions):
        if i == 0:
            result.append(0 if pred in ["A", "B"] else int(pred))
        else:
            if pred in ["A", "B"]:
                result.append(result[-1])
            else:
                result.append(result[-1] + int(pred))

    return "|".join(map(str, result))

In [ ]:
submission = {"subtaskID": [], "datapointID": [], "answer": []}

model.eval()
for patches_list, datapoint_id in test_loader:
    patches = patches_list[0]  # Unwrap batch dimension

    if patches.numel() == 0:  # No patches found
        pred_str = ""
    else:
        patches = patches.to(device)
        if len(patches.shape) == 3:  # Single patch case
            patches = patches.unsqueeze(0)

        with torch.no_grad():
            pred = model(patches)
            y_pred = pred.argmax(dim=1).cpu().numpy()
            labels = train_dataset.get_labels(y_pred)
            pred_str = encode_prediction(labels)

    submission["subtaskID"].append(1)
    submission["datapointID"].append(datapoint_id[0])
    submission["answer"].append(pred_str)

In [ ]:
submission = pd.DataFrame(submission)
submission.head()

In [ ]:
submission.to_csv("submission_eval.csv", index=False)  # 실제 judge(eval_img)용

In [ ]:
# === 우리 채점기용: held-out 합성 strip(eval_local) 예측 → submission.csv (subtaskID,datapointID,answer) ===
# 같은 trained model + extract_patches + encode_prediction 으로 eval_local 에 적용.
ev_local = pd.read_csv('data/eval_local.csv')
local_tf = transforms.Compose([transforms.ToPILImage(), transforms.Resize((48,48)),
                               transforms.ToTensor(), transforms.Normalize(mean=[0.5], std=[0.5])])
rows = []
model.eval()
for _, r in ev_local.iterrows():
    img = cv.imread('data/' + r['datapointID'], cv.IMREAD_GRAYSCALE)
    patches = extract_patches(img)
    if patches:
        xb = torch.stack([local_tf(p) for p in patches]).to(device)
        with torch.no_grad():
            yp = model(xb).argmax(dim=1).cpu().numpy()
        ans = encode_prediction(train_dataset.get_labels(yp))   # 이미 '|'-join 된 문자열 반환
    else:
        ans = ''
    rows.append({'subtaskID': 1, 'datapointID': r['datapointID'], 'answer': ans})
pd.DataFrame(rows).to_csv('submission.csv', index=False)
print('wrote submission.csv (eval_local, 우리 채점기용)', len(rows))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)